# MBA Engenharia de Dados — DataOps
## Controle de Qualidade de Dados com Great Expectations (GX Core 0.18)

**Aluno:** Vinicius Martins Cordeiro  
**E-mail:** vmartinscordeiro@gmail.com  
**Disciplina:** DataOps — FIAP  
**Data:** Maio de 2026  

---

## Contexto

Este notebook implementa testes de qualidade de dados usando **Great Expectations (GX Core)**,
aplicados sobre o dataset `curso` — tabela com dados academicos de alunos do MBA.

O dataset replica os problemas reais identificados durante o Lab de DataOps:
- Ambiguidade semantica: nota zero significa `nao cursou` OU `tirou zero`?
- Inconsistencia de valores ausentes: MAT_1 usa `0`, MAT_4 usa `NULL`
- Campo INGLES sem dominio padronizado (`sim`, `SIM`, `S`, `yes`, string vazia)
- Datas em formato inconsistente (`YYYY-MM-DD` vs `DD/MM/YYYY`)

## Categorias de Testes Implementadas

| # | Categoria | O que valida |
|---|-----------|-------------|
| 1 | **Schema** | Colunas presentes e tipos de dados corretos |
| 2 | **Volume** | Quantidade de registros dentro do esperado |
| 3 | **Valores Numericos e Datas** | Ranges, media, desvio padrao, datas validas |
| 4 | **Formato** | Nulos, regex, comprimento de strings |
| 5 | **Unicidade** | Chave primaria, proporcao de valores distintos |
| 6 | **Integridade Referencial** | Dominio de valores, consistencia entre colunas |

In [1]:
# Instalar dependencias (executar uma vez)
%pip install great-expectations pandas numpy --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import great_expectations as gx
import pandas as pd
import numpy as np
import warnings
from great_expectations.dataset import PandasDataset

warnings.filterwarnings('ignore')

print('Great Expectations : ' + gx.__version__)
print('Pandas             : ' + pd.__version__)
print('NumPy              : ' + np.__version__)

Great Expectations : 0.18.22
Pandas             : 2.3.3
NumPy              : 1.26.4


---
## 1. Dataset — Tabela `curso`

O dataset simula a tabela `curso` do banco MySQL utilizado no Lab de DataOps da FIAP.
Os problemas de qualidade sao introduzidos **propositalmente** para que os testes os detectem.

### Estrutura

| Coluna | Tipo | Observacao |
|--------|------|------------|
| MATRICULA | int | Identificador unico do aluno |
| NOME | str | Nome completo — pode ter homonimos |
| DATA_MATRICULA | str | Data de matricula — ISO ou formato incorreto |
| NOTA_MAT_1..4 | float | Nota 0-10; MAT_4 tem NULLs; zeros em MAT_1 sao ambiguos |
| REPROVACOES_MAT_1..4 | int | Numero de reprovacoes por materia |
| INGLES | str | Proficiencia em ingles — multiplas grafias inconsistentes |

In [3]:
np.random.seed(42)
n = 200

nomes = [
    'Ana Silva', 'Bruno Costa', 'Carla Mendes', 'Diego Lima', 'Elena Rocha',
    'Fabio Dias', 'Gabriela Nunes', 'Henrique Santos', 'Isabela Faria', 'Joao Alves',
    'Karen Pinto', 'Lucas Moreira', 'Marina Souza', 'Nelson Vieira', 'Olivia Castro',
    'Paulo Ramos', 'Renata Gomes', 'Sergio Teixeira', 'Tania Barros', 'Vitor Cunha'
]

df = pd.DataFrame({'MATRICULA': range(1001, 1001 + n)})
df['NOME'] = np.random.choice(nomes, n)  # duplicatas intencionais (homonimos)

# MAT_1: zeros ambiguos (nao cursou vs. nota zero real)
notas1 = np.random.uniform(3.5, 10.0, n)
notas1[np.random.choice(n, 15, replace=False)] = 0.0
df['NOTA_MAT_1'] = np.round(notas1, 1)
df['NOTA_MAT_2'] = np.round(np.clip(np.random.normal(6.5, 1.8, n), 0.0, 10.0), 1)
df['NOTA_MAT_3'] = np.round(np.clip(np.random.normal(7.0, 1.5, n), 0.0, 10.0), 1)

# MAT_4: 30 alunos ainda nao cursaram (NULL intencionais)
notas4 = np.round(np.clip(np.random.normal(6.8, 1.8, n), 0.0, 10.0), 1)
s4 = pd.Series(notas4, dtype=float)
s4.iloc[np.random.choice(n, 30, replace=False)] = np.nan
df['NOTA_MAT_4'] = s4.values

df['REPROVACOES_MAT_1'] = np.random.choice([0, 1, 2, 3], n, p=[0.75, 0.15, 0.07, 0.03])
df['REPROVACOES_MAT_2'] = np.random.choice([0, 1, 2, 3], n, p=[0.78, 0.13, 0.06, 0.03])
df['REPROVACOES_MAT_3'] = np.random.choice([0, 1, 2, 3], n, p=[0.80, 0.12, 0.05, 0.03])
df['REPROVACOES_MAT_4'] = np.random.choice([0, 1, 2, 3], n, p=[0.82, 0.11, 0.05, 0.02])

# INGLES: multiplas grafias inconsistentes (problema de qualidade real do lab)
ingles_vals = ['Sim', 'sim', 'SIM', 'S', 'Nao', 'NAO', 'N', 'nao', 'yes', None, '']
ingles_prob = [0.22, 0.08, 0.05, 0.04, 0.22, 0.07, 0.04, 0.05, 0.03, 0.10, 0.10]
df['INGLES'] = np.random.choice(ingles_vals, n, p=ingles_prob)

# DATA_MATRICULA: maioria em formato ISO, 10 datas em formato incorreto
dates_ok = [
    (pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(d))).strftime('%Y-%m-%d')
    for d in np.random.randint(0, 730, n - 10)
]
dates_bad = [
    '25/12/2024', '01/02/2023', '15/06/2024', '30/11/2023', '20/03/2024',
    '10/07/2024', '05/01/2025', '28/02/2024', '17/08/2023', '11/09/2024'
]
all_dates = dates_ok + dates_bad
np.random.shuffle(all_dates)
df['DATA_MATRICULA'] = all_dates

print('Dataset criado: ' + str(len(df)) + ' registros, ' + str(len(df.columns)) + ' colunas')
print()
for col in df.columns:
    nulls = df[col].isna().sum()
    print('  ' + col + ' (' + str(df[col].dtype) + ') — ' + str(nulls) + ' nulos')

Dataset criado: 200 registros, 12 colunas

  MATRICULA (int64) — 0 nulos
  NOME (object) — 0 nulos
  NOTA_MAT_1 (float64) — 0 nulos
  NOTA_MAT_2 (float64) — 0 nulos
  NOTA_MAT_3 (float64) — 0 nulos
  NOTA_MAT_4 (float64) — 30 nulos
  REPROVACOES_MAT_1 (int32) — 0 nulos
  REPROVACOES_MAT_2 (int32) — 0 nulos
  REPROVACOES_MAT_3 (int32) — 0 nulos
  REPROVACOES_MAT_4 (int32) — 0 nulos
  INGLES (object) — 16 nulos
  DATA_MATRICULA (object) — 0 nulos


In [4]:
print('Amostra do dataset:')
print(df[['MATRICULA', 'NOME', 'DATA_MATRICULA', 'NOTA_MAT_1', 'NOTA_MAT_4', 'INGLES']].head(10).to_string())
print()
print('Distribuicao de INGLES (problema de qualidade):')
print(df['INGLES'].fillna('NULL').value_counts().to_string())
print()
print('Estatisticas das notas:')
print(df[['NOTA_MAT_1', 'NOTA_MAT_2', 'NOTA_MAT_3', 'NOTA_MAT_4']].describe().round(2).to_string())

Amostra do dataset:
   MATRICULA             NOME DATA_MATRICULA  NOTA_MAT_1  NOTA_MAT_4 INGLES
0       1001   Gabriela Nunes     2024-07-12         8.1         6.0    sim
1       1002      Vitor Cunha     2023-10-13         0.0         8.4    nao
2       1003    Olivia Castro     2024-07-16         5.8         4.3    Sim
3       1004      Karen Pinto     2023-02-26         4.1         7.7       
4       1005  Henrique Santos     2023-01-19         9.6         4.9   None
5       1006   Gabriela Nunes     2023-11-23         6.1         9.1    NAO
6       1007     Tania Barros     2023-04-14         6.9         6.3    Nao
7       1008      Karen Pinto     2023-10-04         8.9         7.0   None
8       1009      Karen Pinto     2024-01-30         7.9         7.5    Sim
9       1010       Diego Lima     2024-10-18         8.3         3.3    Sim

Distribuicao de INGLES (problema de qualidade):
INGLES
Nao     49
Sim     48
        17
NAO     17
NULL    16
sim     12
SIM     12
S       11


---
## 2. Great Expectations — Conceitos e Setup

**Great Expectations (GX Core)** e o framework open source mais popular para qualidade de dados.

### Conceitos fundamentais

| Conceito | Descricao |
|----------|-----------|
| **Expectation** | Regra atomica sobre os dados (ex: coluna nao pode ter nulos) |
| **ExpectationSuite** | Conjunto de expectativas agrupadas por tema/dataset |
| **Validator** | Executa as expectativas contra um batch de dados |
| **ValidationResult** | Resultado da execucao — passa/falha por expectativa |
| **Data Docs** | Relatorio HTML gerado automaticamente com os resultados |

### API usada neste notebook

Usamos a classe `PandasDataset` — wrapper do DataFrame que adiciona todos os metodos
`expect_xxx()` diretamente ao objeto. E a forma mais direta e educacional de usar o GX.

```python
validator = PandasDataset(df)
validator.expect_table_row_count_to_be_between(min_value=100, max_value=500)
results = validator.validate()
```

In [5]:
def executar_suite(expectativas_fn, dataframe):
    """
    Cria um PandasDataset, aplica as expectativas via funcao callback
    e retorna o ValidationResult completo.
    """
    validator = PandasDataset(dataframe)
    expectativas_fn(validator)
    return validator.validate()


def exibir_resultados(results, titulo):
    """Exibe o ValidationResult de forma legivel no console."""
    sep = '=' * 70
    passou = sum(1 for r in results.results if r.success)
    falhou = len(results.results) - passou
    status = 'APROVADO' if results.success else 'REPROVADO'
    print(sep)
    print('  ' + titulo)
    print(sep)
    print('  Status  : ' + status)
    print('  Total   : ' + str(len(results.results)))
    print('  Passou  : ' + str(passou))
    print('  Falhou  : ' + str(falhou))
    print(sep)
    for r in results.results:
        st = 'PASSOU' if r.success else 'FALHOU'
        tipo = r.expectation_config.expectation_type
        col = r.expectation_config.kwargs.get('column', 'tabela')
        print()
        print('  [' + st + '] ' + tipo)
        print('           coluna: ' + str(col))
        if not r.success and r.result:
            obs = r.result.get('observed_value', 'N/A')
            print('           valor observado: ' + str(obs))
    print()


print('Funcoes auxiliares definidas: executar_suite(), exibir_resultados()')

Funcoes auxiliares definidas: executar_suite(), exibir_resultados()


---
## Categoria 1 — SCHEMA

**Objetivo:** garantir que as colunas esperadas existam e tenham os tipos corretos.

Mudancas nao anunciadas de schema sao um dos problemas mais comuns em pipelines.
Uma coluna renomeada upstream pode quebrar toda a cadeia downstream sem aviso.

**Expectativas GX usadas:**
- `expect_table_columns_to_match_set` — conjunto de colunas bate com o esperado
- `expect_column_to_exist` — coluna especifica existe na tabela
- `expect_column_values_to_be_of_type` — tipo da coluna e o esperado

In [6]:
colunas_esperadas = [
    'MATRICULA', 'NOME', 'DATA_MATRICULA',
    'NOTA_MAT_1', 'NOTA_MAT_2', 'NOTA_MAT_3', 'NOTA_MAT_4',
    'REPROVACOES_MAT_1', 'REPROVACOES_MAT_2', 'REPROVACOES_MAT_3', 'REPROVACOES_MAT_4',
    'INGLES'
]

def schema_expectations(v):
    # Conjunto completo de colunas deve estar presente
    v.expect_table_columns_to_match_set(column_set=colunas_esperadas)
    # Colunas criticas devem existir individualmente
    v.expect_column_to_exist('MATRICULA')
    v.expect_column_to_exist('NOME')
    v.expect_column_to_exist('NOTA_MAT_1')
    v.expect_column_to_exist('NOTA_MAT_4')
    v.expect_column_to_exist('INGLES')
    v.expect_column_to_exist('DATA_MATRICULA')
    # Tipos de dados esperados (pandas dtype)
    v.expect_column_values_to_be_of_type('MATRICULA', type_='int64')
    v.expect_column_values_to_be_of_type('NOTA_MAT_1', type_='float64')
    v.expect_column_values_to_be_of_type('NOTA_MAT_2', type_='float64')
    v.expect_column_values_to_be_of_type('NOME', type_='object')
    v.expect_column_values_to_be_of_type('INGLES', type_='object')

results_schema = executar_suite(schema_expectations, df)
exibir_resultados(results_schema, 'CATEGORIA 1 — SCHEMA')

  CATEGORIA 1 — SCHEMA
  Status  : APROVADO
  Total   : 12
  Passou  : 12
  Falhou  : 0

  [PASSOU] expect_table_columns_to_match_set
           coluna: tabela

  [PASSOU] expect_column_to_exist
           coluna: MATRICULA

  [PASSOU] expect_column_values_to_be_of_type
           coluna: MATRICULA

  [PASSOU] expect_column_to_exist
           coluna: NOME

  [PASSOU] expect_column_values_to_be_of_type
           coluna: NOME

  [PASSOU] expect_column_to_exist
           coluna: NOTA_MAT_1

  [PASSOU] expect_column_values_to_be_of_type
           coluna: NOTA_MAT_1

  [PASSOU] expect_column_to_exist
           coluna: NOTA_MAT_4

  [PASSOU] expect_column_to_exist
           coluna: INGLES

  [PASSOU] expect_column_values_to_be_of_type
           coluna: INGLES

  [PASSOU] expect_column_to_exist
           coluna: DATA_MATRICULA

  [PASSOU] expect_column_values_to_be_of_type
           coluna: NOTA_MAT_2



---
## Categoria 2 — VOLUME

**Objetivo:** verificar se a quantidade de registros esta dentro dos limites esperados.

Testes de volume detectam:
- Carga incompleta (menos linhas que o esperado — pipeline falhou silenciosamente)
- Duplicacao acidental (mais linhas — join explodiu)
- Tabela zerada (pipeline nao processou nada)

**Expectativas GX usadas:**
- `expect_table_row_count_to_be_between` — numero de linhas no intervalo esperado

In [7]:
def volume_expectations(v):
    # Dataset deve ter entre 100 e 500 alunos
    v.expect_table_row_count_to_be_between(min_value=100, max_value=500)

results_volume = executar_suite(volume_expectations, df)
exibir_resultados(results_volume, 'CATEGORIA 2 — VOLUME')

print('Total de registros: ' + str(len(df)))
print('Alunos sem NOTA_MAT_4 (nao cursaram ainda): ' + str(df['NOTA_MAT_4'].isna().sum()))
print('Alunos com NOTA_MAT_1 = 0 (ambiguos): ' + str((df['NOTA_MAT_1'] == 0.0).sum()))

  CATEGORIA 2 — VOLUME
  Status  : APROVADO
  Total   : 1
  Passou  : 1
  Falhou  : 0

  [PASSOU] expect_table_row_count_to_be_between
           coluna: tabela

Total de registros: 200
Alunos sem NOTA_MAT_4 (nao cursaram ainda): 30
Alunos com NOTA_MAT_1 = 0 (ambiguos): 15


---
## Categoria 3 — VALORES NUMERICOS E DATAS

**Objetivo:** validar que numeros e datas estao dentro dos intervalos esperados e que
medidas estatisticas (media, desvio padrao) sao coerentes com o historico.

Casos detectados:
- Nota acima de 10 (impossivel no sistema — corrupcao de dados)
- Data de matricula anterior ao inicio do curso
- Media de notas muito abaixo do historico (carregamento errado)

**Expectativas GX usadas:**
- `expect_column_values_to_be_between` — cada valor no range
- `expect_column_max_to_be_between` — maximo da coluna no range
- `expect_column_min_to_be_between` — minimo da coluna no range
- `expect_column_mean_to_be_between` — media no range
- `expect_column_stdev_to_be_between` — desvio padrao no range

In [8]:
def numeric_expectations(v):
    # Notas devem estar entre 0 e 10
    v.expect_column_values_to_be_between('NOTA_MAT_1', min_value=0.0, max_value=10.0)
    v.expect_column_values_to_be_between('NOTA_MAT_2', min_value=0.0, max_value=10.0)
    v.expect_column_values_to_be_between('NOTA_MAT_3', min_value=0.0, max_value=10.0)
    v.expect_column_values_to_be_between('NOTA_MAT_4', min_value=0.0, max_value=10.0)
    # Maximo e minimo esperados para MAT_1
    v.expect_column_max_to_be_between('NOTA_MAT_1', min_value=8.0, max_value=10.0)
    v.expect_column_min_to_be_between('NOTA_MAT_1', min_value=0.0, max_value=1.0)
    # Media em faixa razoavel (historico do curso)
    v.expect_column_mean_to_be_between('NOTA_MAT_2', min_value=4.0, max_value=9.0)
    v.expect_column_mean_to_be_between('NOTA_MAT_3', min_value=4.0, max_value=9.0)
    # Desvio padrao: nao pode ser zero (turma identica) nem extremo (dados corrompidos)
    v.expect_column_stdev_to_be_between('NOTA_MAT_2', min_value=0.5, max_value=4.0)
    v.expect_column_stdev_to_be_between('NOTA_MAT_3', min_value=0.5, max_value=4.0)
    # Reprovacoes nao podem ser negativas
    v.expect_column_values_to_be_between('REPROVACOES_MAT_1', min_value=0, max_value=10)
    v.expect_column_values_to_be_between('REPROVACOES_MAT_2', min_value=0, max_value=10)
    # Datas dentro do periodo do curso (comparacao lexicografica funciona para ISO YYYY-MM-DD)
    # Datas no formato DD/MM/YYYY vao FALHAR pois '25/12/2024' > '2025-12-31' lexicograficamente
    v.expect_column_values_to_be_between(
        'DATA_MATRICULA', min_value='2023-01-01', max_value='2025-12-31'
    )

results_numeric = executar_suite(numeric_expectations, df)
exibir_resultados(results_numeric, 'CATEGORIA 3 — VALORES NUMERICOS E DATAS')

print('Datas com formato incorreto (DD/MM/YYYY):')
mask_bad = ~df['DATA_MATRICULA'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
print(df.loc[mask_bad, 'DATA_MATRICULA'].values)

  CATEGORIA 3 — VALORES NUMERICOS E DATAS
  Status  : REPROVADO
  Total   : 13
  Passou  : 12
  Falhou  : 1

  [PASSOU] expect_column_values_to_be_between
           coluna: NOTA_MAT_1

  [PASSOU] expect_column_max_to_be_between
           coluna: NOTA_MAT_1

  [PASSOU] expect_column_min_to_be_between
           coluna: NOTA_MAT_1

  [PASSOU] expect_column_values_to_be_between
           coluna: NOTA_MAT_2

  [PASSOU] expect_column_mean_to_be_between
           coluna: NOTA_MAT_2

  [PASSOU] expect_column_stdev_to_be_between
           coluna: NOTA_MAT_2

  [PASSOU] expect_column_values_to_be_between
           coluna: NOTA_MAT_3

  [PASSOU] expect_column_mean_to_be_between
           coluna: NOTA_MAT_3

  [PASSOU] expect_column_stdev_to_be_between
           coluna: NOTA_MAT_3

  [PASSOU] expect_column_values_to_be_between
           coluna: NOTA_MAT_4

  [PASSOU] expect_column_values_to_be_between
           coluna: REPROVACOES_MAT_1

  [PASSOU] expect_column_values_to_be_between
   

---
## Categoria 4 — FORMATO

**Objetivo:** validar que os dados seguem o formato esperado:
ausencia de nulos onde nao e permitido, comprimento de strings, padroes regex.

Testes de formato sao cruciais para:
- Garantir campos obrigatorios preenchidos
- Validar formatos de data, CPF, email, codigo postal
- Detectar strings vazias mascarando ausencia de informacao

**Expectativas GX usadas:**
- `expect_column_values_to_not_be_null` — campo obrigatorio
- `expect_column_value_lengths_to_be_between` — comprimento da string no range
- `expect_column_values_to_match_regex` — valor segue o padrao
- `expect_column_values_to_not_match_regex` — valor nao pode seguir o padrao

In [9]:
def format_expectations(v):
    # Campos obrigatorios sem nulos
    v.expect_column_values_to_not_be_null('MATRICULA')
    v.expect_column_values_to_not_be_null('NOME')
    v.expect_column_values_to_not_be_null('NOTA_MAT_1')
    v.expect_column_values_to_not_be_null('NOTA_MAT_2')
    v.expect_column_values_to_not_be_null('NOTA_MAT_3')
    # NOTA_MAT_4 tem NULLs intencionais — DEVE FALHAR para evidenciar o problema
    v.expect_column_values_to_not_be_null('NOTA_MAT_4')
    # NOME deve ter comprimento razoavel (nome completo)
    v.expect_column_value_lengths_to_be_between('NOME', min_value=5, max_value=60)
    # DATA_MATRICULA deve seguir formato ISO YYYY-MM-DD
    # DEVE FALHAR: ha 10 datas no formato DD/MM/YYYY
    v.expect_column_values_to_match_regex(
        'DATA_MATRICULA',
        regex=r'^\d{4}-\d{2}-\d{2}$'
    )
    # INGLES nao pode ser string vazia (diferente de NULL — ambos sao problemas diferentes)
    # DEVE FALHAR: ha strings vazias no campo
    v.expect_column_values_to_not_match_regex('INGLES', regex=r'^$')

results_format = executar_suite(format_expectations, df)
exibir_resultados(results_format, 'CATEGORIA 4 — FORMATO')

print('Analise de nulls e strings vazias em INGLES:')
print('  Valores nulos (None/NaN): ' + str(df['INGLES'].isna().sum()))
print('  Strings vazias: ' + str((df['INGLES'].fillna('X') == '').sum()))
print()
print('Analise de NOTA_MAT_4:')
print('  Valores nulos: ' + str(df['NOTA_MAT_4'].isna().sum()) +
      ' alunos que ainda nao cursaram MAT_4')

  CATEGORIA 4 — FORMATO
  Status  : REPROVADO
  Total   : 9
  Passou  : 6
  Falhou  : 3

  [PASSOU] expect_column_values_to_not_be_null
           coluna: MATRICULA

  [PASSOU] expect_column_values_to_not_be_null
           coluna: NOME

  [PASSOU] expect_column_value_lengths_to_be_between
           coluna: NOME

  [PASSOU] expect_column_values_to_not_be_null
           coluna: NOTA_MAT_1

  [PASSOU] expect_column_values_to_not_be_null
           coluna: NOTA_MAT_2

  [PASSOU] expect_column_values_to_not_be_null
           coluna: NOTA_MAT_3

  [FALHOU] expect_column_values_to_not_be_null
           coluna: NOTA_MAT_4
           valor observado: N/A

  [FALHOU] expect_column_values_to_match_regex
           coluna: DATA_MATRICULA
           valor observado: N/A

  [FALHOU] expect_column_values_to_not_match_regex
           coluna: INGLES
           valor observado: N/A

Analise de nulls e strings vazias em INGLES:
  Valores nulos (None/NaN): 16
  Strings vazias: 17

Analise de NOTA_MA

---
## Categoria 5 — UNICIDADE

**Objetivo:** garantir unicidade onde e esperada (chave primaria) e medir a proporcao
de valores distintos em colunas que podem ter repeticoes controladas.

Casos de uso:
- MATRICULA e chave primaria: deve ser 100% unica
- NOME pode ter duplicatas (homonimos), mas a proporcao deve ser baixa e conhecida
- INGLES deve ter poucos valores distintos (dominio finito)

**Expectativas GX usadas:**
- `expect_column_values_to_be_unique` — todos os valores distintos
- `expect_column_proportion_of_unique_values_to_be_between` — % de unicos no range
- `expect_column_unique_value_count_to_be_between` — numero de distintos no range
- `expect_compound_columns_to_be_unique` — combinacao de colunas unica

In [10]:
def uniqueness_expectations(v):
    # MATRICULA e chave primaria: 100% unica
    v.expect_column_values_to_be_unique('MATRICULA')
    # NOME tem homonimos: proporcao de unicos deve ser baixa (~20 nomes para 200 alunos = 10%)
    v.expect_column_proportion_of_unique_values_to_be_between(
        'NOME', min_value=0.05, max_value=0.25
    )
    # INGLES: dominio com poucos valores distintos (mesmo com inconsistencias)
    v.expect_column_unique_value_count_to_be_between('INGLES', min_value=2, max_value=20)
    # Combinacao MATRICULA + NOME deve ser unica (sem duplicatas completas de registro)
    v.expect_compound_columns_to_be_unique(column_list=['MATRICULA', 'NOME'])

results_uniqueness = executar_suite(uniqueness_expectations, df)
exibir_resultados(results_uniqueness, 'CATEGORIA 5 — UNICIDADE')

print('Analise de unicidade:')
print('  Total de registros    : ' + str(len(df)))
print('  Matriculas unicas     : ' + str(df['MATRICULA'].nunique()))
print('  Nomes unicos          : ' + str(df['NOME'].nunique()) +
      ' (de ' + str(len(df)) + ' registros)')
print('  Valores distintos INGLES: ' + str(df['INGLES'].fillna('NULL').nunique()))

  CATEGORIA 5 — UNICIDADE
  Status  : APROVADO
  Total   : 4
  Passou  : 4
  Falhou  : 0

  [PASSOU] expect_column_values_to_be_unique
           coluna: MATRICULA

  [PASSOU] expect_column_proportion_of_unique_values_to_be_between
           coluna: NOME

  [PASSOU] expect_column_unique_value_count_to_be_between
           coluna: INGLES

  [PASSOU] expect_compound_columns_to_be_unique
           coluna: tabela

Analise de unicidade:
  Total de registros    : 200
  Matriculas unicas     : 200
  Nomes unicos          : 20 (de 200 registros)
  Valores distintos INGLES: 11


---
## Categoria 6 — INTEGRIDADE REFERENCIAL

**Objetivo:** verificar se os valores pertencem a um dominio valido e se a relacao
entre colunas e consistente com as regras de negocio.

Dois niveis de validacao:
1. **Dominio de valores**: campo INGLES deve ter apenas valores padronizados
2. **Consistencia entre colunas**: regras de negocio que cruzam duas ou mais colunas

Demonstramos o ciclo completo:  
`dado cru → teste FALHA → normalizacao → dado limpo → teste PASSA`

**Expectativas GX usadas:**
- `expect_column_values_to_be_in_set` — valor pertence ao dominio permitido
- `expect_column_values_to_not_be_in_set` — valor nao pertence ao dominio proibido
- Verificacao pandas para regras de negocio cruzadas entre colunas

In [11]:
# --- 6a: INGLES com dado cru — DEVE FALHAR ---
dominio_padrao = ['Sim', 'Nao']

def ref_raw_expectations(v):
    v.expect_column_values_to_be_in_set('INGLES', value_set=dominio_padrao)

results_ref_raw = executar_suite(ref_raw_expectations, df)
exibir_resultados(
    results_ref_raw,
    'CATEGORIA 6a — INTEGRIDADE REF: INGLES cru (esperado FALHAR)'
)

print('Valores encontrados em INGLES (dado cru):')
print(df['INGLES'].fillna('NULL').value_counts().to_string())

  CATEGORIA 6a — INTEGRIDADE REF: INGLES cru (esperado FALHAR)
  Status  : REPROVADO
  Total   : 1
  Passou  : 0
  Falhou  : 1

  [FALHOU] expect_column_values_to_be_in_set
           coluna: INGLES
           valor observado: N/A

Valores encontrados em INGLES (dado cru):
INGLES
Nao     49
Sim     48
        17
NAO     17
NULL    16
sim     12
SIM     12
S       11
N        8
nao      6
yes      4


In [12]:
# --- Normalizacao: padronizar o campo INGLES ---
def normalizar_ingles(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 'Nao_Informado'
    val_str = str(val).strip()
    if val_str == '':
        return 'Nao_Informado'
    val_up = val_str.upper()
    if val_up in ['SIM', 'S', 'YES', 'Y', '1']:
        return 'Sim'
    if val_up in ['NAO', 'N', 'NO', '0']:
        return 'Nao'
    return 'Nao_Informado'

df_clean = df.copy()
df_clean['INGLES'] = df_clean['INGLES'].apply(normalizar_ingles)

print('Distribuicao apos normalizacao:')
print(df_clean['INGLES'].value_counts().to_string())

Distribuicao apos normalizacao:
INGLES
Sim              87
Nao              80
Nao_Informado    33


In [13]:
# --- 6b: INGLES apos normalizacao — DEVE PASSAR ---
dominio_limpo = ['Sim', 'Nao', 'Nao_Informado']
grafias_invalidas = ['', 'sim', 'SIM', 'S', 'nao', 'NAO', 'N', 'yes']

def ref_clean_expectations(v):
    v.expect_column_values_to_be_in_set('INGLES', value_set=dominio_limpo)
    v.expect_column_values_to_not_be_in_set('INGLES', value_set=grafias_invalidas)

results_ref_clean = executar_suite(ref_clean_expectations, df_clean)
exibir_resultados(
    results_ref_clean,
    'CATEGORIA 6b — INTEGRIDADE REF: INGLES normalizado (esperado PASSAR)'
)

  CATEGORIA 6b — INTEGRIDADE REF: INGLES normalizado (esperado PASSAR)
  Status  : APROVADO
  Total   : 2
  Passou  : 2
  Falhou  : 0

  [PASSOU] expect_column_values_to_be_in_set
           coluna: INGLES

  [PASSOU] expect_column_values_to_not_be_in_set
           coluna: INGLES



In [14]:
# --- 6c: Verificacao cruzada entre colunas (regras de negocio) ---
sep = '=' * 70
print(sep)
print('  CATEGORIA 6c — INTEGRIDADE REF: verificacao cruzada entre colunas')
print(sep)

# Regra 1: aluno com reprovacao em MAT_1 e nota > 0 deve ter nota < 7.0
# (se nota >= 7 seria aprovado — inconsistencia)
mask1 = (
    (df['REPROVACOES_MAT_1'] > 0) &
    (df['NOTA_MAT_1'] > 0.0) &
    (df['NOTA_MAT_1'] >= 7.0)
)
inc1 = df[mask1]
print()
print('  Regra 1: aluno com reprovacao em MAT_1 deve ter nota < 7.0')
if len(inc1) > 0:
    print('  [FALHOU] ' + str(len(inc1)) + ' registros inconsistentes:')
    print(inc1[['MATRICULA', 'NOME', 'NOTA_MAT_1', 'REPROVACOES_MAT_1']].head(5).to_string())
else:
    print('  [PASSOU] Nenhuma inconsistencia encontrada')

# Regra 2: aluno sem nota em MAT_4 (nao cursou) nao pode ter reprovacoes em MAT_4
mask2 = df['NOTA_MAT_4'].isna() & (df['REPROVACOES_MAT_4'] > 0)
inc2 = df[mask2]
print()
print('  Regra 2: aluno sem nota em MAT_4 nao pode ter reprovacoes em MAT_4')
if len(inc2) > 0:
    print('  [FALHOU] ' + str(len(inc2)) + ' registros inconsistentes:')
    print(inc2[['MATRICULA', 'NOME', 'NOTA_MAT_4', 'REPROVACOES_MAT_4']].head(5).to_string())
else:
    print('  [PASSOU] Nenhuma inconsistencia encontrada')

# Regra 3: ambiguidade semantica de nota zero
zero_sem_reprov = (df['NOTA_MAT_1'] == 0.0) & (df['REPROVACOES_MAT_1'] == 0)
zero_com_reprov = (df['NOTA_MAT_1'] == 0.0) & (df['REPROVACOES_MAT_1'] > 0)
print()
print('  Regra 3: analise de ambiguidade em NOTA_MAT_1 = 0')
print('    Provavelmente nao cursou (nota=0 sem reprovacao): ' + str(df[zero_sem_reprov].shape[0]))
print('    Provavelmente cursou e tirou zero (nota=0 com reprovacao): ' + str(df[zero_com_reprov].shape[0]))
print()
print('  Conclusao: nota 0 sem reprovacao deve ser convertida para NULL.')
print('  GX nao detecta ambiguidade semantica sozinho — requer regra de negocio explicita.')
print()

  CATEGORIA 6c — INTEGRIDADE REF: verificacao cruzada entre colunas

  Regra 1: aluno com reprovacao em MAT_1 deve ter nota < 7.0
  [FALHOU] 17 registros inconsistentes:
    MATRICULA            NOME  NOTA_MAT_1  REPROVACOES_MAT_1
0        1001  Gabriela Nunes         8.1                  1
11       1012    Carla Mendes         7.0                  2
15       1016     Bruno Costa         9.9                  1
18       1019   Lucas Moreira        10.0                  1
19       1020    Renata Gomes         9.8                  2

  Regra 2: aluno sem nota em MAT_4 nao pode ter reprovacoes em MAT_4
  [FALHOU] 4 registros inconsistentes:
     MATRICULA           NOME  NOTA_MAT_4  REPROVACOES_MAT_4
106       1107    Elena Rocha         NaN                  1
146       1147  Isabela Faria         NaN                  1
162       1163    Vitor Cunha         NaN                  1
178       1179    Bruno Costa         NaN                  1

  Regra 3: analise de ambiguidade em NOTA_MAT_1 =

---
## Sumario Consolidado

In [15]:
todos = [
    (results_schema,      'SCHEMA'),
    (results_volume,      'VOLUME'),
    (results_numeric,     'VALORES NUMERICOS E DATAS'),
    (results_format,      'FORMATO'),
    (results_uniqueness,  'UNICIDADE'),
    (results_ref_raw,     'INTEGRIDADE REF — INGLES cru'),
    (results_ref_clean,   'INTEGRIDADE REF — INGLES normalizado'),
]

sep = '=' * 70
print(sep)
print('  SUMARIO CONSOLIDADO DE QUALIDADE DE DADOS')
print(sep)

total_passou = 0
total_falhou = 0

for results, nome in todos:
    passou = sum(1 for r in results.results if r.success)
    falhou = len(results.results) - passou
    total_passou += passou
    total_falhou += falhou
    st = 'OK ' if results.success else 'NOK'
    barra = str(passou) + '/' + str(len(results.results))
    print('  [' + st + ']  ' + nome.ljust(40) + barra)

print('-' * 70)
total = total_passou + total_falhou
pct = round(100.0 * total_passou / total, 1) if total > 0 else 0.0
print('  TOTAL: ' + str(total_passou) + '/' + str(total) +
      ' expectativas passaram (' + str(pct) + '%)')
print(sep)
print()
print('Falhas esperadas (demonstram o valor dos testes):')
print('  - NOTA_MAT_4 com nulls: alunos que nao cursaram MAT_4 ainda')
print('  - DATA_MATRICULA fora do formato ISO: 10 datas em DD/MM/YYYY')
print('  - DATA_MATRICULA fora do range: formato errado quebra comparacao')
print('  - INGLES fora do dominio padrao: multiplas grafias inconsistentes')
print('  - INGLES com strings vazias: ausencia nao tratada como NULL')

  SUMARIO CONSOLIDADO DE QUALIDADE DE DADOS
  [OK ]  SCHEMA                                  12/12
  [OK ]  VOLUME                                  1/1
  [NOK]  VALORES NUMERICOS E DATAS               12/13
  [NOK]  FORMATO                                 6/9
  [OK ]  UNICIDADE                               4/4
  [NOK]  INTEGRIDADE REF — INGLES cru            0/1
  [OK ]  INTEGRIDADE REF — INGLES normalizado    2/2
----------------------------------------------------------------------
  TOTAL: 37/42 expectativas passaram (88.1%)

Falhas esperadas (demonstram o valor dos testes):
  - NOTA_MAT_4 com nulls: alunos que nao cursaram MAT_4 ainda
  - DATA_MATRICULA fora do formato ISO: 10 datas em DD/MM/YYYY
  - DATA_MATRICULA fora do range: formato errado quebra comparacao
  - INGLES fora do dominio padrao: multiplas grafias inconsistentes
  - INGLES com strings vazias: ausencia nao tratada como NULL


---
## Etapa 2 — Modelo de Maturidade

### Pergunta
**A partir de qual modelo de maturidade deveríamos ter testes para garantir a qualidade dos dados no nosso pipeline?**

---

### Resposta

Os testes de qualidade devem ser introduzidos a partir do **Nível 2 — Gerenciado/Repetível**.

| Nivel | Nome | Pipeline | Testes de Qualidade |
|-------|------|----------|--------------------|
| 1 | Reativo / Ad-hoc | SQL manual, phpMyAdmin | Nao — problemas descobertos por acaso |
| **2** | **Gerenciado / Repetivel** | **ETL automatizado, versionado** | **SIM — ponto critico de introducao** |
| 3 | Definido / Padronizado | dbt, Airflow, CI/CD | Sim — integrados ao pipeline |
| 4 | Mensuravel / Quantitativo | SLAs, observabilidade | Sim — monitoramento continuo |
| 5 | Otimizado | ML, self-service, automacao | Sim — qualidade como produto |

### Por que o Nivel 2 e o ponto critico?

No **Nivel 1** (cenario do Lab), os dados sao consultados manualmente.
Problemas de qualidade sao tolerados pois o analista ve os dados diretamente.

No **Nivel 2**, a carga e transformacao comecam a ser automatizadas.
E exatamente aqui que problemas de qualidade ganham **escala**:
um dado ruim que entrava manualmente passa a se propagar para todos os
consumidores downstream sem que ninguem perceba.

Sem testes no Nivel 2:
- Dashboards exibem metricas incorretas sem alerta
- Modelos de ML sao treinados com dados corrompidos
- A origem do erro e dificil de rastrear (sem linhagem)

### O que o Great Expectations entrega nesse contexto

O GX Core permite implementar todas as categorias de teste deste notebook
e integra-las ao pipeline de dados:

- **Expressivo**: `expect_column_values_to_be_between` e legivel por nao-tecnicos
- **Iterativo**: facil de adicionar regras conforme o dado e o negocio evoluem
- **Integravel**: chamado em Airflow DAGs, dbt tests, GitHub Actions CI/CD
- **Documentavel**: gera Data Docs HTML com historico de validacoes

### Referencia de maturidade

Baseado em **DAMA-DMBOK** e no modelo de maturidade de DataOps (Gartner/DCAM).
O cenario do lab (SQL manual + phpMyAdmin + Superset sobre dado bruto) = **Nivel 1**.
A introducao de testes automatizados com GX Core = primeiro passo para o **Nivel 2**.

---

## Referencias

- Great Expectations — The Beginner's Guide to Data Quality Tests (PDF fornecido no curso)
- Great Expectations Documentation: https://docs.greatexpectations.io
- DAMA-DMBOK: Data Management Body of Knowledge
- Repositorio do trabalho: https://github.com/tonanuvem/datacatalog

---
## 3. Estrutura de Projeto — Great Expectations Filesystem

Além da execução em memória (células anteriores), o GX Core pode persistir
toda a configuração e os resultados em disco, gerando um **projeto GX** reutilizável.

### Estrutura gerada automaticamente

```
Trabalho_DataOps/
├── dataops_qualidade_dados.ipynb   ← este notebook
├── data/
│   └── curso.csv                   ← dataset exportado
└── gx/
    ├── great_expectations.yml      ← configuração principal do projeto GX
    ├── expectations/
    │   └── suite_qualidade_dados.json  ← regras salvas (ExpectationSuite)
    └── uncommitted/
        └── data_docs/
            └── local_site/
                └── index.html      ← relatório HTML (Data Docs)
```

### Por que manter os arquivos?

| Arquivo | Papel no pipeline |
|---------|------------------|
| `great_expectations.yml` | Raiz do projeto — aponta datasources e sites |
| `suite_qualidade_dados.json` | Regras versionáveis no Git — evoluem com o negócio |
| `index.html` (Data Docs) | Relatório auditável gerado a cada execução |

O **Data Docs** é o artefato chave para comunicar qualidade de dados
para stakeholders não-técnicos: visualiza passes e falhas por expectativa,
com histórico de execuções.


In [16]:
import os
import great_expectations as gx

# Garante que o notebook rode a partir da pasta Trabalho_DataOps/
NB_DIR = os.path.dirname(os.path.abspath('__file__'))
TRABALHO_DIR = r'c:\Users\vmart\OneDrive\Documentos\MBA_FIAP_Aulas\7ABDR\Aula_DataOps\Trabalho_DataOps'
os.chdir(TRABALHO_DIR)
print('Diretório de trabalho:', os.getcwd())

# Salvar dataset como CSV
csv_path = os.path.join(TRABALHO_DIR, 'data', 'curso.csv')
df.to_csv(csv_path, index=False)
print('Dataset salvo em:', csv_path)
print('  Registros:', len(df))

# Criar contexto GX filesystem (gera a pasta gx/ com great_expectations.yml)
ctx = gx.get_context(mode='file', project_root_dir=TRABALHO_DIR)
print('\nContexto GX criado em:', TRABALHO_DIR)

# Registrar datasource pandas
ds = ctx.sources.add_or_update_pandas('pandas_source')
asset = ds.add_dataframe_asset('curso_asset')
batch_request = asset.build_batch_request(dataframe=df)
print('Datasource e asset configurados.')

# Criar/atualizar suite de expectativas
suite_name = 'suite_qualidade_dados'
suite = ctx.add_or_update_expectation_suite(suite_name)
validator = ctx.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# ── Adicionar todas as expectativas ao validator ──────────────────────────────
colunas_esperadas = [
    'MATRICULA', 'NOME', 'DATA_MATRICULA',
    'NOTA_MAT_1', 'NOTA_MAT_2', 'NOTA_MAT_3', 'NOTA_MAT_4',
    'REPROVACOES_MAT_1', 'REPROVACOES_MAT_2', 'REPROVACOES_MAT_3', 'REPROVACOES_MAT_4',
    'INGLES'
]

# SCHEMA
validator.expect_table_columns_to_match_set(column_set=colunas_esperadas)
validator.expect_column_to_exist('MATRICULA')
validator.expect_column_to_exist('NOTA_MAT_4')
validator.expect_column_values_to_be_of_type('MATRICULA', type_='int64')
validator.expect_column_values_to_be_of_type('NOTA_MAT_1', type_='float64')

# VOLUME
validator.expect_table_row_count_to_be_between(min_value=100, max_value=500)

# VALORES NUMÉRICOS E DATAS
validator.expect_column_values_to_be_between('NOTA_MAT_1', min_value=0.0, max_value=10.0)
validator.expect_column_values_to_be_between('NOTA_MAT_2', min_value=0.0, max_value=10.0)
validator.expect_column_values_to_be_between('NOTA_MAT_3', min_value=0.0, max_value=10.0)
validator.expect_column_values_to_be_between('NOTA_MAT_4', min_value=0.0, max_value=10.0)
validator.expect_column_mean_to_be_between('NOTA_MAT_2', min_value=4.0, max_value=9.0)
validator.expect_column_stdev_to_be_between('NOTA_MAT_2', min_value=0.5, max_value=4.0)
validator.expect_column_values_to_be_between(
    'DATA_MATRICULA', min_value='2023-01-01', max_value='2025-12-31'
)

# FORMATO
validator.expect_column_values_to_not_be_null('MATRICULA')
validator.expect_column_values_to_not_be_null('NOME')
validator.expect_column_values_to_not_be_null('NOTA_MAT_4')  # vai falhar
validator.expect_column_value_lengths_to_be_between('NOME', min_value=5, max_value=60)
validator.expect_column_values_to_match_regex('DATA_MATRICULA', regex=r'^\d{4}-\d{2}-\d{2}$')
validator.expect_column_values_to_not_match_regex('INGLES', regex=r'^$')

# UNICIDADE
validator.expect_column_values_to_be_unique('MATRICULA')
validator.expect_column_proportion_of_unique_values_to_be_between(
    'NOME', min_value=0.05, max_value=0.25
)
validator.expect_column_unique_value_count_to_be_between('INGLES', min_value=2, max_value=20)

# INTEGRIDADE REFERENCIAL
validator.expect_column_values_to_be_in_set('INGLES', value_set=['Sim', 'Nao'])  # vai falhar

# Salvar a suite em disco (gx/expectations/suite_qualidade_dados.json)
validator.save_expectation_suite(discard_failed_expectations=False)
print('\nExpectationSuite salva em: gx/expectations/' + suite_name + '.json')

# Listar arquivos criados
import pathlib
gx_dir = pathlib.Path(TRABALHO_DIR) / 'gx'
print('\nArquivos do projeto GX:')
for p in sorted(gx_dir.rglob('*')):
    if p.is_file():
        rel = p.relative_to(pathlib.Path(TRABALHO_DIR))
        print('  ', rel)


Diretório de trabalho: c:\Users\vmart\OneDrive\Documentos\MBA_FIAP_Aulas\7ABDR\Aula_DataOps\Trabalho_DataOps
Dataset salvo em: c:\Users\vmart\OneDrive\Documentos\MBA_FIAP_Aulas\7ABDR\Aula_DataOps\Trabalho_DataOps\data\curso.csv
  Registros: 200

Contexto GX criado em: c:\Users\vmart\OneDrive\Documentos\MBA_FIAP_Aulas\7ABDR\Aula_DataOps\Trabalho_DataOps
Datasource e asset configurados.


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]


ExpectationSuite salva em: gx/expectations/suite_qualidade_dados.json

Arquivos do projeto GX:
   gx\.gitignore
   gx\expectations\.ge_store_backend_id
   gx\expectations\suite_qualidade_dados.json
   gx\great_expectations.yml
   gx\plugins\custom_data_docs\styles\data_docs_custom_styles.css
   gx\uncommitted\config_variables.yml
   gx\uncommitted\data_docs\local_site\expectations\suite_qualidade_dados.html
   gx\uncommitted\data_docs\local_site\index.html
   gx\uncommitted\data_docs\local_site\static\fonts\HKGrotesk\HKGrotesk-Bold.otf
   gx\uncommitted\data_docs\local_site\static\fonts\HKGrotesk\HKGrotesk-BoldItalic.otf
   gx\uncommitted\data_docs\local_site\static\fonts\HKGrotesk\HKGrotesk-Italic.otf
   gx\uncommitted\data_docs\local_site\static\fonts\HKGrotesk\HKGrotesk-Light.otf
   gx\uncommitted\data_docs\local_site\static\fonts\HKGrotesk\HKGrotesk-LightItalic.otf
   gx\uncommitted\data_docs\local_site\static\fonts\HKGrotesk\HKGrotesk-Medium.otf
   gx\uncommitted\data_docs\local_

In [17]:
# Executar validação completa no contexto GX filesystem
result = validator.validate()

passou_gx = sum(1 for r in result.results if r.success)
falhou_gx = len(result.results) - passou_gx
status_gx = 'APROVADO' if result.success else 'REPROVADO'

print('=' * 60)
print('  VALIDAÇÃO GX — CONTEXTO FILESYSTEM')
print('=' * 60)
print('  Status :', status_gx)
print('  Total  :', len(result.results))
print('  Passou :', passou_gx)
print('  Falhou :', falhou_gx)
print()

# Gerar Data Docs (HTML)
ctx.build_data_docs()

# Localizar o index.html
import pathlib, os
TRABALHO_DIR = r'c:\Users\vmart\OneDrive\Documentos\MBA_FIAP_Aulas\7ABDR\Aula_DataOps\Trabalho_DataOps'
data_docs_index = (
    pathlib.Path(TRABALHO_DIR)
    / 'gx' / 'uncommitted' / 'data_docs' / 'local_site' / 'index.html'
)

if data_docs_index.exists():
    print('Data Docs gerado com sucesso!')
    print('Arquivo:', data_docs_index)
    print()
    # Abrir no browser
    import webbrowser
    webbrowser.open(data_docs_index.as_uri())
    print('Relatório HTML aberto no browser.')
else:
    print('Arquivo index.html não encontrado — verifique o diretório gx/')

# Exibir árvore final da pasta Trabalho_DataOps
print()
print('Estrutura final do projeto:')
root = pathlib.Path(TRABALHO_DIR)
for p in sorted(root.rglob('*')):
    if p.is_file():
        rel = p.relative_to(root)
        parts = rel.parts
        indent = '  ' + '  ' * (len(parts) - 1)
        print(indent + parts[-1])


Calculating Metrics:   0%|          | 0/58 [00:00<?, ?it/s]

  VALIDAÇÃO GX — CONTEXTO FILESYSTEM
  Status : REPROVADO
  Total  : 23
  Passou : 18
  Falhou : 5

Data Docs gerado com sucesso!
Arquivo: c:\Users\vmart\OneDrive\Documentos\MBA_FIAP_Aulas\7ABDR\Aula_DataOps\Trabalho_DataOps\gx\uncommitted\data_docs\local_site\index.html

Relatório HTML aberto no browser.

Estrutura final do projeto:
    curso.csv
  dataops_qualidade_dados.ipynb
    .gitignore
      .ge_store_backend_id
      suite_qualidade_dados.json
    great_expectations.yml
          data_docs_custom_styles.css
      config_variables.yml
            suite_qualidade_dados.html
          index.html
                HKGrotesk-Bold.otf
                HKGrotesk-BoldItalic.otf
                HKGrotesk-Italic.otf
                HKGrotesk-Light.otf
                HKGrotesk-LightItalic.otf
                HKGrotesk-Medium.otf
                HKGrotesk-MediumItalic.otf
                HKGrotesk-Regular.otf
                HKGrotesk-SemiBold.otf
                HKGrotesk-SemiBoldItalic.ot